In [ ]:
import mlflow
from openai import OpenAI
from mlflow.genai import scorer
from mlflow.genai.scorers import Correctness, Guidelines

In [ ]:
# Specify the tracking URI for the MLflow server.
mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment("llm-evaluation")

In [ ]:
client = OpenAI()

# To eval something (agent, workflow, model etc ...) you need:

# 1. A dataset
eval_dataset = [
    {
        "inputs": {"question": "In which year was Palermo FC founded?"},
        "expectations": {"expected_response": "1900"},
    },
    {
        "inputs": {"question": "What are the colours of Palermo FC?"},
        "expectations": {"expected_response": "Rosa & Nero"},
    },
    {
        "inputs": {"question": "What is the name of Palermo FC's home stadium?"},
        "expectations": {"expected_response": "Stadio Renzo Barbera"},
    },
    {
        "inputs": {"question": "Who's the current head coach of Palermo FC?"},
        "expectations": {"expected_response": "Filippo Inzaghi"},
    },
]


@scorer
def is_concise(outputs: str) -> bool:
    """Evaluate if the answer is concise (less than 5 words)"""
    return len(outputs.split()) <= 5


# 2. A list of scorers
scorers = [
    Correctness(),
    Guidelines(name="is_italian", guidelines="The answer must be in Italian"),
    is_concise,
]


# 3. A predict function
def predict_fn(question: str) -> str:

    response = client.responses.create(
        model="gpt-6-luna",
        instructions="You are a helpful assistant. Reply in Italian and with less than 5 words.",
        input=question,
    )
    return response.output_text

In [ ]:
# Run the evaluation
results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict_fn,
    scorers=scorers,
)